In [1]:

import pandas as pd

data = {
    "student_id": list(range(101, 121)),
    "attendance_pct": [
        92, 97, 78, 85, 68, 94, 82, 98, 75, 88,
        91, 70, 86, 95, 80, 73, 89, 96, 77, 84
    ],
    "study_hours_week": [
        12, 15, 7, 9, 5, 13, 8, 16, 6, 10,
        11, 4, 9, 14, 7, 5, 12, 15, 6, 8
    ],
    "previous_marks": [
        78, 90, 65, 72, 55, 85, 70, 94, 62, 80,
        83, 50, 74, 89, 68, 58, 81, 92, 60, 76
    ],
    "final_marks": [
        82, 93, 68, 78, 59, 88, 73, 96, 65, 84,
        86, 54, 77, 92, 71, 61, 85, 95, 64, 80
    ]
}

df = pd.DataFrame(data)

df.to_csv("student_performance_prediction.csv", index=False)

print("Dataset created successfully!")
print("Dataset shape:", df.shape)
display(df.head())


Dataset created successfully!
Dataset shape: (20, 5)


,student_id,attendance_pct,study_hours_week,previous_marks,final_marks
0,101,92,12,78,82
1,102,97,15,90,93
2,103,78,7,65,68
3,104,85,9,72,78
4,105,68,5,55,59


In [2]:

# 1. Check dataset dimensions
print("Dataset shape:", df.shape)

# 2. Check column data types
print("\nData types:")
print(df.dtypes)

# 3. Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# 4. Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# 5. Check invalid values
print("\nInvalid attendance:",
      (~df["attendance_pct"].between(0, 100)).sum())

print("Invalid previous marks:",
      (~df["previous_marks"].between(0, 100)).sum())

print("Invalid final marks:",
      (~df["final_marks"].between(0, 100)).sum())

print("Invalid study hours:",
      (df["study_hours_week"] < 0).sum())

# 6. View statistical summary
display(df.describe())


Dataset shape: (20, 5)

Data types:
student_id          int64
attendance_pct      int64
study_hours_week    int64
previous_marks      int64
final_marks         int64
dtype: object

Missing values:
student_id          0
attendance_pct      0
study_hours_week    0
previous_marks      0
final_marks         0
dtype: int64

Duplicate rows: 0

Invalid attendance: 0
Invalid previous marks: 0
Invalid final marks: 0
Invalid study hours: 0


,student_id,attendance_pct,study_hours_week,previous_marks,final_marks
count,20.00000,20.000000,20.000000,20.000000,20.000000
mean,110.50000,84.900000,9.600000,74.100000,77.550000
std,5.91608,9.267373,3.719083,12.940756,12.700912
min,101.00000,68.000000,4.000000,50.000000,54.000000
25%,105.75000,77.750000,6.750000,64.250000,67.250000
50%,110.50000,85.500000,9.000000,75.000000,79.000000
75%,115.25000,92.500000,12.250000,83.500000,86.500000
max,120.00000,98.000000,16.000000,94.000000,96.000000


In [3]:

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# Select input features and target
X = df[["attendance_pct", "study_hours_week", "previous_marks"]]
y = df["final_marks"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Baseline model: always predicts the training-set average
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

# Machine-learning model
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Compare model performance
results = pd.DataFrame({
    "Model": ["Baseline", "Linear Regression"],
    "MAE": [
        mean_absolute_error(y_test, baseline_predictions),
        mean_absolute_error(y_test, predictions)
    ],
    "RMSE": [
        mean_squared_error(y_test, baseline_predictions) ** 0.5,
        mean_squared_error(y_test, predictions) ** 0.5
    ],
    "R2": [
        r2_score(y_test, baseline_predictions),
        r2_score(y_test, predictions)
    ]
})

display(results.round(2))


,Model,MAE,RMSE,R2
0,Baseline,13.40,14.18,-0.02
1,Linear Regression,0.49,0.59,1.00


In [4]:

# Compare actual and predicted marks
comparison = pd.DataFrame({
    "Actual_Marks": y_test.values,
    "Predicted_Marks": predictions.round(2)
})

comparison["Absolute_Error"] = (
    comparison["Actual_Marks"] - comparison["Predicted_Marks"]
).abs().round(2)

display(comparison)

# Calculate average prediction error
print(
    "Mean Absolute Error:",
    round(comparison["Absolute_Error"].mean(), 2)
)

# Find the largest prediction error
print(
    "Largest prediction error:",
    comparison["Absolute_Error"].max()
)


,Actual_Marks,Predicted_Marks,Absolute_Error
0,82,81.56,0.44
1,95,94.76,0.24
2,61,61.95,0.95
3,93,92.94,0.06
4,65,65.78,0.78


Mean Absolute Error: 0.49
Largest prediction error: 0.95


The Linear Regression model was evaluated against a simple mean-prediction baseline using a held-out test set. MAE and RMSE measure prediction error, while R² provides a relative measure of fit. Because the dataset is small and synthetic, the results are exploratory and cannot establish real-world predictive reliability.